In [1]:
import time, csv, math

def load_csv(fp):
    with open(fp, newline='', encoding='utf-8') as f:
        r = [dict(x) for x in csv.DictReader(f)]
    assert isinstance(r, list), "load_csv must return a list"
    assert len(r) > 0, "CSV file is empty – check the path"
    assert isinstance(r[0], dict), "Each record must be a dictionary"
    return r

def cast_numerics(recs):
    ints = ['Age','Years_Experience','Salary_Before_AI','Salary_After_AI','Work_Hours_Per_Week','Job_Satisfaction']
    for rec in recs:
        for f in ints: rec[f] = int(rec[f])
        rec['Productivity_Change_%'] = float(rec['Productivity_Change_%'])
        rec['Salary_Change'] = rec['Salary_After_AI'] - rec['Salary_Before_AI']
    sample = recs[0]
    assert isinstance(sample['Age'], int), "Age must be int"
    assert isinstance(sample['Salary_Before_AI'], int), "Salary_Before_AI must be int"
    assert isinstance(sample['Productivity_Change_%'], float), "Productivity must be float"
    assert 'Salary_Change' in sample, "Salary_Change must be derived"
    return recs

def build_index(recs, key):
    assert len(recs) > 0, "records must not be empty"
    assert key in recs[0], f"key_field '{key}' not found in records"
    idx = {}
    for r in recs: idx.setdefault(r[key], []).append(r)
    total = sum(len(v) for v in idx.values())
    assert total == len(recs), f"Index mismatch: {total} indexed vs {len(recs)} total"
    assert all(isinstance(k, str) for k in idx.keys()), "All keys must be strings"
    return idx

def build_full_index(recs):
    idx = {k: build_index(recs, f) for k, f in [('by_industry','Industry'),('by_job_role','Job_Role'),
            ('by_risk','Automation_Risk'),('by_ai_level','AI_Adoption_Level'),('by_job_status','Job_Status')]}
    assert set(idx.keys()) == {'by_industry','by_job_role','by_risk','by_ai_level','by_job_status'}, \
        "Full index must have exactly five sub-indexes"
    return idx

class TreeNode:
    def __init__(self, name):
        self.name, self.children, self.employee_count = name, [], 0
        self.salary_changes, self.replaced_count = [], 0
    def find_child(self, name):
        return next((c for c in self.children if c.name == name), None)
    def avg_salary_change(self):
        return sum(self.salary_changes)/len(self.salary_changes) if self.salary_changes else 0
    def replacement_rate(self):
        return (self.replaced_count/self.employee_count*100) if self.employee_count else 0

class IndustryJobTree:
    def __init__(self): self.industries = {}
    def build_tree(self, recs):
        for r in recs:
            ind, role, status = r['Industry'], r['Job_Role'], r['Job_Status']
            pct = (r['Salary_After_AI']-r['Salary_Before_AI'])/r['Salary_Before_AI']*100
            inode = self.industries.setdefault(ind, TreeNode(ind))
            jnode = inode.find_child(role)
            if jnode is None:
                jnode = TreeNode(role); inode.children.append(jnode)
            for n in (inode, jnode):
                n.employee_count += 1; n.salary_changes.append(pct)
                if status == 'Replaced': n.replaced_count += 1
    def display_tree(self):
        print("\n"+"="*60+"\nINDUSTRY → JOB ROLE HIERARCHY\n"+"="*60)
        for iname in sorted(self.industries):
            n = self.industries[iname]
            print(f"\n📁 {iname}\n    Employees: {n.employee_count:,}\n"
                  f"    Replaced: {n.replaced_count} ({n.replacement_rate():.1f}%)\n"
                  f"    Avg Salary Change: {n.avg_salary_change():+.1f}%")
            for j in sorted(n.children, key=lambda x: x.name):
                print(f"    └── {j.name}\n         Employees: {j.employee_count:,}\n"
                      f"         Replaced: {j.replaced_count} ({j.replacement_rate():.1f}%)\n"
                      f"         Avg Salary Change: {j.avg_salary_change():+.1f}%")
    def get_industry_summary(self):
        s = [{'Industry':n.name,'Total_Employees':n.employee_count,'Replaced_Count':n.replaced_count,
              'Replacement_Rate_%':n.replacement_rate(),'Avg_Salary_Change_%':n.avg_salary_change()}
             for n in self.industries.values()]
        return quicksort(s, key=lambda x: -x['Replacement_Rate_%'])
    def get_high_risk_jobs(self, threshold_pct=30):
        hr = []
        for inode in self.industries.values():
            for j in inode.children:
                rr = j.replacement_rate()
                if rr >= threshold_pct:
                    hr.append({'Industry':inode.name,'Job_Role':j.name,'Employees':j.employee_count,
                                'Replaced':j.replaced_count,'Replacement_Rate_%':rr,
                                'Avg_Salary_Change_%':j.avg_salary_change()})
        return quicksort(hr, key=lambda x: -x['Replacement_Rate_%'])

def quicksort(lst, key=lambda x: x):
    assert isinstance(lst, list), "Input to quicksort must be a list"
    if len(lst) <= 1: return lst[:]
    p = key(lst[len(lst)//2])
    left  = [x for x in lst if key(x) < p]
    mid   = [x for x in lst if key(x) == p]
    right = [x for x in lst if key(x) > p]
    out = quicksort(left, key) + mid + quicksort(right, key)
    assert len(out) == len(lst), "QuickSort must not lose or duplicate elements"
    for i in range(len(out)-1):
        assert key(out[i]) <= key(out[i+1]), f"QuickSort: element at {i} is greater than element at {i+1}"
    return out

def selection_sort(lst, key=lambda x: x):
    assert isinstance(lst, list), "Input must be a list"
    assert len(lst) > 0, "List must not be empty"
    orig_len = len(lst)
    n = len(lst)
    for i in range(n):
        m = i
        for j in range(i+1, n):
            if key(lst[j]) < key(lst[m]): m = j
        lst[i], lst[m] = lst[m], lst[i]
    assert len(lst) == orig_len, "Selection sort must not change list length"
    for i in range(len(lst)-1):
        assert key(lst[i]) <= key(lst[i+1]), f"Selection sort: order violation at index {i}"
    return lst

def binary_search(s, target, key=lambda x: x):
    assert isinstance(s, list), "sorted_lst must be a list"
    n = len(s)
    if n > 1:
        assert key(s[0]) <= key(s[n-1]), "Binary search precondition violated: list is not sorted"
    lo, hi = 0, n-1
    while lo <= hi:
        mid = (lo+hi)//2
        v = key(s[mid])
        if v == target:
            assert key(s[mid]) == target, "Binary search: returned index does not match target"
            return mid
        elif v < target: lo = mid+1
        else: hi = mid-1
    assert target not in [key(x) for x in s], "Binary search returned -1 but target IS in list (bug!)"
    return -1

def binary_search_all(s, target, key=lambda x: x):
    idx = binary_search(s, target, key)
    if idx == -1: return []
    res = []
    i = idx
    while i >= 0 and key(s[i]) == target: res.append(i); i -= 1
    i = idx+1
    while i < len(s) and key(s[i]) == target: res.append(i); i += 1
    res = sorted(res)
    assert all(key(s[i]) == target for i in res), "binary_search_all: not all returned indices match target"
    return res

def linear_search_all(recs, field, target):
    assert isinstance(recs, list), "records must be a list"
    assert len(recs) > 0, "records must not be empty"
    assert field in recs[0], f"Field '{field}' not found in records"
    results = [r for r in recs if r[field] == target]
    assert isinstance(results, list), "linear_search_all must return a list"
    assert all(r[field] == target for r in results), "linear_search_all: some results do not match target"
    return results

def mean(v):
    assert len(v) > 0, "mean: input list must not be empty"
    result = sum(v)/len(v)
    assert isinstance(result, (int, float)), "mean must return a number"
    return result

def median(v):
    assert len(v) > 0, "median: input list must not be empty"
    s = quicksort(list(v)); n = len(s); m = n//2
    return (s[m-1]+s[m])/2 if n%2==0 else s[m]

def mode(v):
    assert len(v) > 0, "mode: input list must not be empty"
    freq = {}
    for x in v: freq[x] = freq.get(x,0)+1
    mx = max(freq.values())
    return next(k for k,c in freq.items() if c==mx)

def std_dev(v):
    assert len(v) >= 2, "std_dev: need at least 2 values"
    m = mean(v)
    result = math.sqrt(sum((x-m)**2 for x in v)/len(v))
    assert result >= 0, "Standard deviation must be non-negative"
    return result

def count_by_category(recs, field):
    assert len(recs) > 0, "records must not be empty"
    assert field in recs[0], f"Field '{field}' not in records"
    c = {}
    for r in recs: c[r[field]] = c.get(r[field],0)+1
    assert sum(c.values()) == len(recs), "count_by_category: counts do not sum to total records"
    return c

def avg_by_category(recs, gf, vf):
    assert len(recs) > 0
    assert gf in recs[0]
    assert vf in recs[0]
    b = {}
    for r in recs: b.setdefault(r[gf],[]).append(r[vf])
    averages = {k: mean(v) for k,v in b.items()}
    assert all(isinstance(v, (int, float)) for v in averages.values()), \
        "avg_by_category must return numeric averages"
    return averages

def problem1_salary_analysis(records, index):
    assert 'by_ai_level' in index, "Index must have 'by_ai_level'"
    results = {}
    for level, group in index['by_ai_level'].items():
        sc = [r['Salary_Change'] for r in group]
        jsc = count_by_category(group, 'Job_Status')
        results[level] = {'count':len(group),'mean_salary_change':round(mean(sc),2),
                           'median_salary_change':round(median(sc),2),'std_dev_salary':round(std_dev(sc),2),
                           'job_status_counts':jsc,
                           'replacement_rate_%':round(jsc.get('Replaced',0)/len(group)*100,2)}
    assert set(results.keys()) == {'Low','Medium','High'}, "Must have stats for all three AI adoption levels"
    return results

def problem2_automation_risk(records, index):
    assert 'by_risk' in index, "Index must contain 'by_risk'"
    hr = linear_search_all(records, 'Automation_Risk', 'High')
    assert len(hr) > 0, "There must be at least one high-risk record"
    assert all(r['Automation_Risk'] == 'High' for r in hr), "All collected records must be High automation risk"
    total = len(hr)
    ic, rc = count_by_category(hr,'Industry'), count_by_category(hr,'Job_Role')
    ir = quicksort([(k,v,round(v/total*100,1)) for k,v in ic.items()], key=lambda x:-x[1])
    rr = quicksort([(k,v,round(v/total*100,1)) for k,v in rc.items()], key=lambda x:-x[1])
    assert ir[0][1] >= ir[-1][1], "Industries must be sorted descending by count"
    return {'total_high_risk':total,'industry_ranking':ir,'job_role_ranking':rr}

def time_algorithm(func, *args, repeats=3):
    assert callable(func), "func must be callable"
    assert repeats >= 1, "repeats must be at least 1"
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter(); func(*args); times.append(time.perf_counter()-t0)
    avg = mean(times)
    assert avg >= 0, "Execution time must be non-negative"
    return avg

def benchmark_sorting(records):
    assert len(records) >= 1000, "Need at least 1000 records to benchmark"
    sizes = [50,100,500,1000,2000]
    results = {}
    for size in sizes:
        if size > len(records): break
        subset = [r['Salary_Before_AI'] for r in records[:size]]
        qs = time_algorithm(quicksort, subset[:])
        ss = time_algorithm(selection_sort, subset[:])
        results[size] = {'quicksort_s':round(qs,6),'selection_sort_s':round(ss,6)}
    assert all('quicksort_s' in v for v in results.values()), "Each size result must include quicksort timing"
    return results

def benchmark_search(records):
    assert len(records) >= 1000, "Need at least 1000 records to benchmark"
    sizes = [50,100,500,1000,2000]
    target = records[len(records)//2]['Salary_Before_AI']
    salaries = [r['Salary_Before_AI'] for r in records]
    sorted_sal = quicksort(salaries[:])
    results = {}
    for size in sizes:
        if size > len(records): break
        lin = time_algorithm(linear_search_all, records[:size], 'Salary_Before_AI', target)
        bin_ = time_algorithm(binary_search_all, sorted_sal[:size], target)
        results[size] = {'linear_s':round(lin,6),'binary_s':round(bin_,6)}
    assert all('linear_s' in v for v in results.values()), "Each size result must include linear search timing"
    return results

def run_correctness_tests():
    print("\n"+"="*60+"\n  ALGORITHM CORRECTNESS TESTS (Assertions)\n"+"="*60)
    assert quicksort([38,27,43,3,9,82,10]) == [3,9,10,27,38,43,82]
    assert quicksort([]) == [] and quicksort([5]) == [5]
    print("  [PASS] QuickSort")
    t2 = [64,25,12,22,11]; selection_sort(t2); assert t2 == [11,12,22,25,64]
    print("  [PASS] SelectionSort")
    sn = [2,5,8,12,16,23,38,56,72,91]
    assert binary_search(sn,23) != -1 and binary_search(sn,99) == -1
    print("  [PASS] BinarySearch")
    dl = [1,2,2,3,3,3,4,5]
    assert len(binary_search_all(dl,3)) == 3 and binary_search_all(dl,99) == []
    print("  [PASS] binary_search_all")
    sr = [{'Industry':'IT'},{'Industry':'Finance'},{'Industry':'IT'}]
    f = linear_search_all(sr,'Industry','IT')
    assert len(f) == 2 and linear_search_all(sr,'Industry','Healthcare') == []
    print("  [PASS] LinearSearch")
    nums = [4,8,15,16,23,42]
    assert mean(nums) == sum(nums)/len(nums) and median(nums) == 15.5 and mode([1,2,2,3]) == 2
    assert round(std_dev([2,4,4,4,5,5,7,9]),4) == 2.0
    print("  [PASS] Statistical helpers")
    recs = [{'J':'A'},{'J':'B'},{'J':'A'},{'J':'C'},{'J':'A'}]
    assert count_by_category(recs,'J') == {'A':3,'B':1,'C':1}
    print("  [PASS] count_by_category")
    tr = [{'Industry':'IT'},{'Industry':'Finance'},{'Industry':'IT'}]
    bi = build_index(tr,'Industry')
    assert len(bi['IT']) == 2 and sum(len(v) for v in bi.values()) == 3
    print("  [PASS] build_index")
    print("="*60+"\n  ALL CORRECTNESS TESTS PASSED\n"+"="*60+"\n")

def print_section(title):
    print("\n"+"="*60+f"\n  {title}\n"+"="*60)

def print_table(headers, rows, col_widths=None):
    if col_widths is None:
        col_widths = [max(len(str(h)), max((len(str(r[i])) for r in rows), default=0)) for i,h in enumerate(headers)]
    print("  "+" | ".join(str(h).ljust(w) for h,w in zip(headers,col_widths)))
    print("  "+"-"*(sum(col_widths)+3*(len(headers)-1)))
    for row in rows:
        print("  "+" | ".join(str(c).ljust(w) for c,w in zip(row,col_widths)))

def main():
    FILEPATH = 'ai_job_impact.csv'
    print("\n"+"#"*60+"\n  AI JOB IMPACT – DSA ANALYSIS\n  Data Structure: Dictionary (Hash Map) + Tree\n"
          "  Algorithms: QuickSort | SelectionSort | BinarySearch\n"
          "              LinearSearch | Statistical Aggregation\n"+"#"*60)

    print_section("1. LOADING AND PREPARING DATA")
    records = cast_numerics(load_csv(FILEPATH))
    print(f"  Loaded {len(records)} employee records with {len(records[0])} fields each.")
    print(f"  Fields: {', '.join(records[0].keys())}")

    print_section("2. BUILDING DICTIONARY (HASH MAP) INDEX")
    index = build_full_index(records)
    for name, bucket in index.items():
        print(f"  {name}: {len(bucket)} categories – {sorted(bucket.keys())}")

    print_section("3. BUILDING INDUSTRY-JOB ROLE HIERARCHY TREE")
    tree = IndustryJobTree(); tree.build_tree(records); tree.display_tree()

    print("\n"+"-"*60+"\nHIGH-RISK JOB ROLES (Replacement Rate > 30%)\n"+"-"*60)
    hrj = tree.get_high_risk_jobs(30)
    if hrj:
        print_table(['Industry','Job Role','Employees','Replaced','Replacement Rate','Avg Salary Change'],
            [(h['Industry'],h['Job_Role'],h['Employees'],h['Replaced'],f"{h['Replacement_Rate_%']:.1f}%",
              f"{h['Avg_Salary_Change_%']:+.1f}%") for h in hrj[:10]], col_widths=[15,22,10,9,16,18])
    else:
        print("  No job roles with replacement rate above 30%")

    print("\n"+"-"*60+"\nINDUSTRY SUMMARY (Sorted by Replacement Rate)\n"+"-"*60)
    isum = tree.get_industry_summary()
    print_table(['Industry','Total Employees','Replaced','Replacement Rate','Avg Salary Change'],
        [(s['Industry'],s['Total_Employees'],s['Replaced_Count'],f"{s['Replacement_Rate_%']:.1f}%",
          f"{s['Avg_Salary_Change_%']:+.1f}%") for s in isum], col_widths=[15,16,9,16,18])

    run_correctness_tests()

    print_section("4. PROBLEM 1 – AI ADOPTION vs SALARY & JOB STATUS")
    p1 = problem1_salary_analysis(records, index)
    print("\n  AI Adoption Level | Avg Salary Change | Median | Std Dev | Replaced%")
    print("  "+"-"*75)
    for level in ['Low','Medium','High']:
        d = p1[level]
        print(f"  {level:<18} | {d['mean_salary_change']:>17,.2f} | {d['median_salary_change']:>6,.0f} | "
              f"{d['std_dev_salary']:>7,.0f} | {d['replacement_rate_%']:>9.1f}%")
    print("\n  Job Status Breakdown per AI Adoption Level:")
    for level in ['Low','Medium','High']:
        print(f"    {level}: {p1[level]['job_status_counts']}")

    print_section("5. PROBLEM 2 – AUTOMATION RISK BY INDUSTRY & JOB ROLE")
    p2 = problem2_automation_risk(records, index)
    print(f"\n  Total High-Risk Records : {p2['total_high_risk']}")
    print("\n  Top Industries by High Automation Risk:")
    print_table(['Industry','Count','Share%'], [(r[0],r[1],f"{r[2]}%") for r in p2['industry_ranking']], col_widths=[15,7,8])
    print("\n  Top Job Roles by High Automation Risk:")
    print_table(['Job Role','Count','Share%'], [(r[0],r[1],f"{r[2]}%") for r in p2['job_role_ranking'][:8]], col_widths=[20,7,8])

    print_section("6. SORTED SALARY TABLE (QuickSort – Divide & Conquer)")
    sr = quicksort(records, key=lambda r: r['Salary_Before_AI'])
    for label, subset in [("5 Lowest Salaries Before AI:", sr[:5]), ("5 Highest Salaries Before AI:", sr[-5:])]:
        print(f"\n  {label}")
        print_table(['Employee_ID','Job_Role','Industry','Salary_Before','Salary_After','Change'],
            [(r['Employee_ID'],r['Job_Role'][:18],r['Industry'],f"${r['Salary_Before_AI']:,}",
              f"${r['Salary_After_AI']:,}",f"${r['Salary_Change']:+,}") for r in subset], col_widths=[13,20,14,14,13,8])

    print_section("7. INDUSTRY-LEVEL STATISTICS (Greedy Aggregation)")
    asc = avg_by_category(records,'Industry','Salary_Change')
    apr = avg_by_category(records,'Industry','Productivity_Change_%')
    isorted = quicksort(list(asc.keys()), key=lambda i: -asc[i])
    print_table(['Industry','Avg Salary Change','Avg Productivity Change%'],
        [(i,f"${asc[i]:+,.0f}",f"{apr[i]:+.2f}%") for i in isorted], col_widths=[15,18,24])

    print_section("8. PERFORMANCE BENCHMARKS – Dynamic Analysis")
    print("\n  Sorting Benchmark (QuickSort vs SelectionSort):")
    sb = benchmark_sorting(records)
    print_table(['Input Size','QuickSort (s)','SelectionSort (s)','Speedup Factor'],
        [(size,f"{v['quicksort_s']:.6f}",f"{v['selection_sort_s']:.6f}",
          f"{v['selection_sort_s']/v['quicksort_s']:.1f}x" if v['quicksort_s']>0 else "N/A") for size,v in sb.items()], col_widths=[11,14,18,14])

    print("\n  Search Benchmark (LinearSearch vs BinarySearch):")
    sc = benchmark_search(records)
    print_table(['Input Size','LinearSearch (s)','BinarySearch (s)','Speedup Factor'],
        [(size,f"{v['linear_s']:.6f}",f"{v['binary_s']:.6f}",
          f"{v['linear_s']/v['binary_s']:.1f}x" if v['binary_s']>0 else "N/A") for size,v in sc.items()], col_widths=[11,17,17,14])

    print_section("9. OVERALL DATASET SUMMARY")
    asc_all = [r['Salary_Change'] for r in records]
    aprod = [r['Productivity_Change_%'] for r in records]
    print(f"\n  Total employees analysed : {len(records):,}")
    print(f"\n  Salary Change (Before → After AI):")
    print(f"    Mean    : ${mean(asc_all):+,.2f}\n    Median  : ${median(asc_all):+,.2f}")
    print(f"    Std Dev : ${std_dev(asc_all):,.2f}\n    Min     : ${min(asc_all):+,.0f}\n    Max     : ${max(asc_all):+,.0f}")
    print(f"\n  Productivity Change (%):")
    print(f"    Mean    : {mean(aprod):+.2f}%\n    Median  : {median(aprod):+.2f}%\n    Std Dev : {std_dev(aprod):.2f}%")
    print(f"\n  Job Status Distribution:")
    for status, count in count_by_category(records,'Job_Status').items():
        pct = count/len(records)*100
        print(f"    {status:<12}: {count:>5} ({pct:5.1f}%) {'█'*int(pct//2)}")

    print("\n"+"#"*60+"\n  ANALYSIS COMPLETE\n"+"#"*60)

main()



############################################################
  AI JOB IMPACT – DSA ANALYSIS
  Data Structure: Dictionary (Hash Map) + Tree
  Algorithms: QuickSort | SelectionSort | BinarySearch
              LinearSearch | Statistical Aggregation
############################################################

  1. LOADING AND PREPARING DATA
  Loaded 2000 employee records with 18 fields each.
  Fields: Employee_ID, Age, Gender, Education_Level, Industry, Job_Role, Years_Experience, AI_Adoption_Level, Automation_Risk, Upskilling_Required, Salary_Before_AI, Salary_After_AI, Job_Status, Work_Hours_Per_Week, Remote_Work, Job_Satisfaction, Productivity_Change_%, Salary_Change

  2. BUILDING DICTIONARY (HASH MAP) INDEX
  by_industry: 7 categories – ['Education', 'Finance', 'Healthcare', 'IT', 'Manufacturing', 'Marketing', 'Retail']
  by_job_role: 21 categories – ['Accountant', 'Administrator', 'Auditor', 'Cashier', 'Content Creator', 'Data Analyst', 'DevOps Engineer', 'Digital Marketer', 'Doct